# Voice AI Fundamentals

The real-time voice pipeline: STT, TTS, VAD, turn-taking, latency, orchestration, and telephony.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice, interview-focused for real-time voice
agent roles. Read the concept, run the example, complete the exercises.

## Learning Objectives

- Explain STT vs TTS, streaming vs batch, and the key streaming mechanics (interim results, endpointing).
- Describe VAD and why semantic end-of-turn detection replaced fixed silence timeouts.
- Walk through barge-in handling and its failure modes (echo, backchannels, false interrupts).
- Break down a sub-1s voice-to-voice latency budget stage by stage.
- Compare LiveKit Agents vs Pipecat, and cascaded pipelines vs speech-to-speech models.
- Explain how a phone call (Twilio/PSTN) reaches an AI agent, and how to survive a dropped connection.

## Mental Model

A voice agent is a **full-duplex streaming pipeline** racing a human
expectation: people expect a reply within ~200-300ms of finishing a sentence.
Every stage must stream and overlap — nothing waits for anything to finish:

```
mic audio ──▶ VAD ──▶ STT (interim + final transcripts)
                          │ (end of turn detected)
                          ▼
                     LLM (streams tokens)
                          │ (sentence chunks)
                          ▼
                     TTS (streams audio) ──▶ speaker
        ▲ ...while STT/VAD keep listening for barge-in the whole time
```

The pipeline's job is not just transcribe-think-speak; it is *conversation
management*: knowing when the user is done, when to shut up, and when to
recover.

## Important Concepts

- **STT (speech-to-text)**: streaming over WebSocket for live calls; batch (file upload) only for post-call re-transcription. Interim (`is_final: false`) vs final transcripts; endpointing.
- **TTS (text-to-speech)**: what matters is TTFB (time to first audio byte) on streamed input; sentence-chunked streaming.
- **VAD (voice activity detection)**: "is someone speaking right now" — Silero VAD is the open-source standard (CPU, ~32ms chunks, probability output).
- **Semantic end-of-turn**: models that decide "is the user *done*" from content + acoustics, not just silence (LiveKit Turn Detector, Pipecat Smart Turn).
- **Barge-in**: user interrupts agent speech; pipeline must stop TTS, cancel LLM, and truncate history to what was actually heard.
- **Latency budget**: target 500-800ms voice-to-voice; LLM TTFT is usually the biggest line item.
- **Cascaded vs speech-to-speech**: STT→LLM→TTS chain vs native audio models (OpenAI Realtime, Gemini Live).

## Need-To-Know Coverage Checklist

- [x] Streaming vs batch STT; interim vs final transcripts; endpointing and UtteranceEnd.
- [x] Current STT landscape: Deepgram Nova-3, OpenAI gpt-4o-transcribe, AssemblyAI Universal-Streaming.
- [x] Current TTS landscape and TTFB numbers: Cartesia Sonic, ElevenLabs Flash, Deepgram Aura.
- [x] Silero VAD; semantic turn detection; smart endpointing.
- [x] Barge-in flow and failure modes: echo false-interrupts, backchannels, cough recovery.
- [x] The canonical latency budget table.
- [x] LiveKit Agents vs Pipecat architecture; Vapi/Retell managed alternatives.
- [x] Twilio Media Streams, ConversationRelay, SIP trunking to LiveKit; mu-law 8kHz reality.
- [x] Dropped-connection handling: ICE restart, state checkpointing, persist-then-callback.

## Deep Study Notes

### STT

- **Deepgram Nova-3** is the default for voice agents (default on Vapi, Retell,
  LiveKit, Pipecat): ~5-7% WER, sub-300ms typical streaming latency,
  ~$0.0077/min, keyterm prompting for domain vocabulary.
- **OpenAI gpt-4o-transcribe**: ~4.1% WER (vs Whisper-v3 ~5.3%) at $0.006/min.
  Whisper itself has no true streaming — which is why it lost the live voice
  market despite ubiquity.
- **AssemblyAI Universal-Streaming**: ~300ms immutable partials; built-in end-of-turn detection.
- Streaming mechanics (Deepgram vocabulary, asked in interviews):
  `interim_results=true` gives provisional transcripts revised as audio
  arrives; `is_final: true` marks stabilized segments; `endpointing=<ms>` gives
  silence-based end-of-speech (`speech_final: true`); `utterance_end_ms` gives
  word-gap-based UtteranceEnd, robust to background noise. Production pattern:
  buffer `is_final` segments, flush on `speech_final` or `UtteranceEnd`.
  Trap: `smart_format=true` can delay finals mid-phone-number.

### TTS

- **Cartesia Sonic** (state-space architecture): fastest, ~40-90ms TTFB, 5-10x
  cheaper than ElevenLabs at volume, built for agents.
- **ElevenLabs Flash v2.5**: ~75ms model inference, ~100-200ms real-world TTFB,
  best-in-class naturalness, priciest. WebSocket input API accepts incremental text.
- **Deepgram Aura-2**: ~200-300ms TTFB; convenient if already on Deepgram.
- **OpenAI gpt-4o-mini-tts**: cheapest (~$0.015/min) but 300-600ms first chunk.
- Pattern: **sentence-chunked streaming** — pipe LLM tokens to TTS at
  sentence/clause boundaries, so TTS TTFB only gates the first sentence.
- Word-level timestamps matter: they let you truncate chat history correctly
  after an interruption (you know exactly what the user heard).

### VAD and end-of-turn

- **Silero VAD** (v5/v6): small neural net, CPU-only, ~32ms chunks, speech
  probability 0-1 with tunable thresholds. First-stage gate in LiveKit and Pipecat.
- The core problem: **silence != done**. Fixed timeouts either cut users off
  mid-thought ("my number is... [pause]") or feel sluggish.
- **LiveKit Turn Detector v1.0** (2026): audio-native, fine-tuned from a 0.5B
  LLM, fuses acoustic + semantic cues, 14 languages; open-weight mini variant.
- **Pipecat Smart Turn v3**: open-source, raw-waveform semantic VAD, ~65ms CPU
  inference; triggered when Silero detects a pause, classifies complete vs
  incomplete turn.
- Managed platforms brand this "smart endpointing".

### Turn-taking and barge-in

Pipelines are full-duplex: STT/VAD run continuously while TTS plays.
Barge-in flow: VAD fires while agent speaking → stop TTS playout → cancel
in-flight LLM generation → **truncate the assistant message in history to what
was actually heard** → treat user speech as a new turn.

Failure modes interviewers probe:
- **Echo false-interrupts**: speaker audio re-enters the mic and the agent
  interrupts itself. Fix: acoustic echo cancellation (built into browser
  WebRTC; speakerphone hardware is the danger case).
- **Backchannels**: "mm-hmm", "yeah" should not stop the agent. Fix:
  `min_interruption_duration` / `min_interruption_words` thresholds.
- **False interruption recovery**: cough triggers VAD but no transcript
  materializes → agent resumes speaking after a timeout (LiveKit behavior).
- **Preemptive generation**: start the LLM on interim transcripts before
  end-of-turn is confirmed; hold TTS until confirmed; discard if the
  transcript changes. Buys back 200-400ms.

### The latency budget (memorize this table)

Human conversational gap is ~200-300ms; production target is **500-800ms
voice-to-voice, sub-800ms at P95**.

| Stage | Typical | Best-in-class |
|---|---|---|
| Capture + VAD/endpoint wait | ~50ms (+300-700ms hidden silence wait) | semantic turn detection |
| STT final transcript | 150-300ms | 60-120ms |
| LLM TTFT | 400-800ms (frontier) — biggest item | 100-250ms (fast tier) |
| TTS TTFB | ~150ms | 40-90ms |
| Network/jitter buffer | 50ms | 20-60ms |

Optimizations: regional co-location, pre-warmed WebSocket connections,
preemptive generation, sentence-chunk TTS, streaming everything, filler
utterances during tool calls. Telephony adds ~100-300ms PSTN overhead.

### Orchestration

- **LiveKit Agents** (infrastructure-first): WebRTC SFU rooms; the agent joins
  a room as a participant. You compose `AgentSession(vad=..., stt=..., llm=...,
  tts=..., turn_detection=...)`; a worker/job model dispatches one agent
  process per room. Self-hostable; LiveKit Cloud for managed scale.
- **Pipecat** (Daily; pipeline-first): you compose a directed graph of frame
  processors; frames (AudioRawFrame, TranscriptionFrame, LLMTextFrame) flow
  through: `Pipeline([transport.input(), stt, llm, tts, transport.output()])`.
  Transport-agnostic (Daily WebRTC, Twilio WebSocket, local); more granular control.
- **Managed**: Vapi (~$0.05/min platform fee, all-in $0.10-0.33/min) and Retell
  (all-in $0.13-0.31/min) — config-over-code; trade speed-to-market for
  lock-in, cost at scale, and limited pipeline control.
- **Speech-to-speech**: OpenAI Realtime API (gpt-realtime; audio tokens ~$32/M
  in, $64/M out; server VAD + semantic VAD modes) and Gemini Live (much
  cheaper: ~$3/M audio in, $12/M out). S2S gives ~300-500ms latency and
  prosody/emotion; cascaded gives model choice per stage, cheaper predictable
  cost, guardrails/observability on visible text, and swap-ability. 2026
  consensus: cascaded dominates telephony/enterprise; hybrid (Realtime model
  as the "LLM" inside LiveKit/Pipecat) is common.

### Telephony

Phone audio is 8kHz 8-bit **mu-law mono** (G.711) — expect degraded STT
accuracy vs 16kHz WebRTC; transcode at the boundary.

1. **Twilio Media Streams**: TwiML `<Connect><Stream>` forks call audio to your
   WebSocket as base64 mu-law frames; bidirectional streams send audio back.
   **ConversationRelay** is the higher-level offering: Twilio runs STT/TTS at
   the edge, you exchange text over WebSocket.
2. **SIP trunking to LiveKit** (canonical): caller dials your Twilio number →
   Twilio Elastic SIP Trunk forwards the INVITE to LiveKit's SIP endpoint →
   inbound trunk auth + dispatch rule → SIP participant joins a room → agent
   worker dispatched to the room. Outbound is the mirror; DTMF and SIP REFER
   transfers supported. Telnyx/Plivo are cheaper PSTN alternatives.

### Dropped connections

- **Transport**: WebRTC handles blips via ICE restart; LiveKit SDKs
  auto-reconnect and resume the same participant session (mobile WiFi-to-
  cellular handoff). WebSocket transports need app-level reconnect.
- **State**: conversation state (history, collected answers, position in the
  flow) must live *outside* the media session — Redis/DB keyed by session id,
  checkpointed every turn — so any worker can rehydrate and resume.
- **PSTN reality**: if the carrier drops the leg there is no resume. Strategy:
  **persist-then-callback** — detect disconnect, persist state, call the user
  back and open with "looks like we got cut off", rehydrating context.
- Hygiene: heartbeats to detect silent death, idempotent tool calls (a dropped
  call must not double-charge), provider failover, post-call webhooks that
  fire regardless of how the call ended.

## Common Failure Modes

- Waiting for final transcripts before doing anything → adds full endpointing delay to every turn.
- Fixed silence timeout as end-of-turn → cuts off thoughtful speakers or feels sluggish.
- No history truncation on barge-in → the agent believes it said things the user never heard.
- No AEC on speakerphone → agent interrupts itself in a loop.
- Blocking tool calls with dead air → users assume the call dropped; use filler speech.
- Conversation state stored only in the worker process → a crash loses the whole interview.
- Ignoring 8kHz mu-law degradation → phone-call WER far worse than demo WER.

## Implementation Notes

- Start from LiveKit Agents or Pipecat; do not hand-roll WebRTC + WebSocket plumbing.
- Choose STT/TTS by *streaming* latency benchmarks (P95, network-inclusive), not batch WER alone.
- Instrument per-stage latency (STT, LLM TTFT, TTS TTFB) from day one; you cannot fix what you cannot attribute.
- Test with real phone audio (8kHz, background noise), not laptop-mic demos.

In [ ]:
"""Simulate a voice turn's latency budget and barge-in truncation logic.

Pure simulation: shows the two calculations interviewers ask you to reason
about — (1) where does the second go in a voice turn, (2) how do you truncate
history when the user interrupts mid-sentence.
"""

# --- 1. Latency budget ---------------------------------------------------
STAGES = [
    ("endpoint wait (silence/semantic)", 250),
    ("STT final transcript", 180),
    ("LLM time-to-first-token", 450),
    ("TTS time-to-first-byte", 90),
    ("network + jitter buffer", 60),
]

total = sum(ms for _, ms in STAGES)
print(f"voice-to-voice: {total}ms  (target: <800ms P95)")
for name, ms in STAGES:
    bar = "#" * (ms // 20)
    print(f"  {name:<36} {ms:>4}ms {bar}")

# --- 2. Barge-in truncation ----------------------------------------------
def truncate_on_interrupt(agent_text: str, word_timestamps: list[tuple[str, int]],
                          interrupted_at_ms: int) -> str:
    """Keep only the words the user actually heard before interrupting.

    word_timestamps: (word, start_ms) pairs from the TTS engine.
    Without this, the chat history claims the agent said things the user
    never heard, and the conversation derails.
    """
    heard = [w for w, start in word_timestamps if start < interrupted_at_ms]
    return " ".join(heard)


tts_words = [("Thanks", 0), ("for", 200), ("sharing", 350), ("that.", 600),
             ("Now", 900), ("let's", 1050), ("talk", 1200), ("about", 1350),
             ("pricing.", 1500)]
history_entry = truncate_on_interrupt("...", tts_words, interrupted_at_ms=1000)
print("\nagent history after barge-in at t=1000ms:")
print(f'  "{history_entry}"  (not the full scripted sentence)')


## Practice

1. Recompute the latency budget replacing the frontier LLM (450ms TTFT) with a
   fast-tier model (150ms) and Cartesia TTS (45ms). What is the new total, and
   which stage is now the bottleneck?
2. Extend `truncate_on_interrupt` to also return whether the interruption was
   likely a backchannel (user speech < 2 words) that should *not* stop the agent.
3. Whiteboard the path of one audio frame from a phone caller to the LLM via
   Twilio SIP trunk → LiveKit, naming each hop.
4. Argue both sides: when would you pick a speech-to-speech model over a
   cascaded pipeline for an interview product? (Hint: the transcript *is* the
   product.)

## Design Checklist

- [ ] Every stage streams; nothing waits for a complete upstream result.
- [ ] Semantic end-of-turn detection, not just a fixed silence timeout.
- [ ] Barge-in stops TTS, cancels LLM, and truncates history via word timestamps.
- [ ] Backchannel and false-interrupt thresholds tuned.
- [ ] Per-stage latency instrumented; P95 voice-to-voice < 800ms.
- [ ] Conversation state checkpointed outside the worker every turn.
- [ ] Persist-then-callback plan for hard PSTN drops.
- [ ] Provider fallbacks (STT/TTS/LLM) with circuit breakers.